# **4. QRC-Mosaic Auxiliary Tool: Dynamic Dual-Eye Animator**
*Creates final dynamic artwork file **animated_pair_of_eyes_natural.gif***

This post-processing notebook is designed to take the required product images from the main project notebook, specifically the generated overlays for:
- **qrc_eye_creator_overlay.png** (Creator Lineages)
- **qrc_eye_geography_overlay.png** (Geographical Origins)
- **qrc_eye_semantic_overlay.png** (VLM-Generated Tech-Centric Thematic Eras)
- **qrc_eye_year_overlay.png** (Historical Year)

...as well as the original binary mask and final calcualted mosaic:
- **qrc_eye_mosaic_soft.png** (Final Calculated QRC Mosaic)
- **target_eye_binary_mask_overlay.png** (Original Target Eye Binary Mask)

It processes these static frames through a crossfading render loop and applies eyelid gating masks to construct a transitional masked dynamic GIF.

To complete the artwork, the engine isolates the internal iris to apply natural saccadic motion, inverts a horizontally mirrored copy next to the original, and synchronises the parallel gaze.

By simultaneously animating the historical pattern layers and the shifting internal irises, the tool makes the two eyes appear to dynamically come to life.

### 1. Dependency Installation:

- Installs required Python libraries (OpenCV, MediaPipe, and Pillow) for image processing and GIF rendering.

In [1]:
!pip install -q mediapipe opencv-python pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 11.3 MB/s eta 0:00:00


###2. Base Animation Engine:

- Loads and scales all input overlay images and final calculated mosaic image to a unified master resolution.
  - qrc_eye_creator_overlay.png (Creator Lineages)
  - qrc_eye_geography_overlay.png (Geographical Origins)
  - qrc_eye_semantic_overlay.png (VLM-Generated Tech-Centric Thematic Eras)
  - qrc_eye_year_overlay.png (Historical Year)
  - qrc_eye_mosaic.png (Final Calculated QRC Mosaic)
- Crossfades transitional layers and applies a basic sine-wave shift to the isolated pupil.
-  Saves animation as **animated_eye_seamless.gif**  

### Amend qrc_eye_semantic_overlay.png:
Amend qrc_eye_semantic_overlay.png to have skin toned background first. The file will become qrc_eye_semantic_skintone.png

In [2]:
!pip install opencv-python numpy

In [3]:
import cv2
import numpy as np

# Load image preserving the Alpha channel (reads actual file headers regardless of .jpg extension)
image = cv2.imread('qrc_eye_semantic_overlay.png', cv2.IMREAD_UNCHANGED)

if image is None:
    print("Error: Could not load image. Make sure it is uploaded to Colab.")
elif image.shape[2] != 4:
    print("Error: Image does not have an Alpha channel.")
else:
    # Split channels and merge RGB back together
    b, g, r, a = cv2.split(image)
    bgr = cv2.merge((b, g, r))

    # Create skin tone background canvas (RGB: 242, 210, 189 -> OpenCV BGR: 189, 210, 242)
    skin_tone_color = np.array([186, 189, 197], dtype=np.uint8)
    skintone_background = np.full_like(bgr, skin_tone_color)

    # Normalize alpha mask
    alpha_factor = a.astype(float) / 255.0
    alpha_factor = np.dstack([alpha_factor] * 3)

    # Blend image over skin tone background
    blended = (bgr.astype(float) * alpha_factor) + (skintone_background.astype(float) * (1 - alpha_factor))
    final_image = blended.astype(np.uint8)

    # Save silently to Colab's local temporary storage as a PNG
    output_filename = 'qrc_eye_semantic_skintone.png'
    cv2.imwrite(output_filename, final_image)
    print(f"Success: Image saved silently to Colab as '{output_filename}'.")

Success: Image saved silently to Colab as 'qrc_eye_semantic_skintone.png'.


In [4]:
import cv2
import numpy as np
import imageio
import math
import glob
import os

# --- 1. CONFIGURATION ---
ORIGINAL_PUPIL_X = 959
ORIGINAL_PUPIL_Y = 782
ORIGINAL_RADIUS = 350
ORIGINAL_MAX_SHIFT = 100
FRAMES_TOTAL = 90 # 90 frames at 30fps = 3 seconds exactly

SCALE = 0.5

PUPIL_X = int(ORIGINAL_PUPIL_X * SCALE)
PUPIL_Y = int(ORIGINAL_PUPIL_Y * SCALE)
RADIUS = int(ORIGINAL_RADIUS * SCALE)
MAX_SHIFT = int(ORIGINAL_MAX_SHIFT * SCALE)

# --- 2. LOAD & RESIZE IMAGES ---
# Define the exact list of files you want to process
required_images = [
    'qrc_eye_creator_overlay.png',
    'qrc_eye_geography_overlay.png',
    'qrc_eye_semantic_skintone.png',
    'qrc_eye_year_overlay.png',
    'qrc_eye_mosaic_soft.png'
]

image_paths = []
for filename in required_images:
    # Check if the file actually exists before adding it to the list
    if os.path.exists(filename):
        image_paths.append(filename)
    else:
        print(f"Warning: '{filename}' not found in the current directory.")

if len(image_paths) == 0:
    raise FileNotFoundError("No images found. Upload them to Colab.")

first_img = cv2.imread(image_paths[0])
target_w = int(first_img.shape[1] * SCALE)
target_h = int(first_img.shape[0] * SCALE)

print(f"Loaded {len(image_paths)} images.")
print(f"Forcing all images to master resolution: {target_w}x{target_h}...")

images = []
for p in image_paths:
    img = cv2.imread(p)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (target_w, target_h))
    images.append(img_resized)

h, w, c = images[0].shape

# --- 3. PREPARE MASKS ---
hard_mask = np.zeros((h, w), dtype=np.uint8)
cv2.circle(hard_mask, (PUPIL_X, PUPIL_Y), RADIUS, 255, -1)
inpaint_mask = cv2.dilate(hard_mask, np.ones((15, 15), np.uint8), iterations=2)

blurred_mask = cv2.GaussianBlur(hard_mask, (81, 81), 0)
mask_float = blurred_mask.astype(float) / 255.0
mask_float = np.expand_dims(mask_float, axis=-1)

output_frames = []
num_images = len(images)
frames_per_transition = FRAMES_TOTAL // num_images if num_images > 0 else FRAMES_TOTAL

# --- 4. RENDER LOOP ---
for i in range(FRAMES_TOTAL):
    img_idx = (i // frames_per_transition) % num_images
    next_idx = (img_idx + 1) % num_images
    local_frame = i % frames_per_transition
    alpha = local_frame / frames_per_transition

    base_frame = cv2.addWeighted(images[img_idx], 1 - alpha, images[next_idx], alpha, 0)

    base_bgr = cv2.cvtColor(base_frame, cv2.COLOR_RGB2BGR)
    bg_inpainted = cv2.inpaint(base_bgr, inpaint_mask, 25, cv2.INPAINT_TELEA)
    bg_inpainted = cv2.cvtColor(bg_inpainted, cv2.COLOR_BGR2RGB)

    progress = i / FRAMES_TOTAL

    # Half-sine wave: seamlessly looks to one side and returns to the center.
    shift_x = int(MAX_SHIFT * math.sin(progress * math.pi))

    M = np.float32([[1, 0, shift_x], [0, 1, 0]])
    shifted_pupil = cv2.warpAffine(base_frame, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    shifted_mask = cv2.warpAffine(mask_float, M, (w, h))
    if len(shifted_mask.shape) == 2:
        shifted_mask = np.expand_dims(shifted_mask, axis=-1)

    final_frame = (bg_inpainted * (1 - shifted_mask) + shifted_pupil * shifted_mask).astype(np.uint8)
    output_frames.append(final_frame)

    if i % 10 == 0:
        print(f"Rendered frame {i}/{FRAMES_TOTAL} (Pupil shifted by {shift_x} pixels)")

# --- 5. EXPORT ---
output_filename = 'animated_eye_seamless.gif'
print("Saving GIF... this may take a moment.")
imageio.mimsave(output_filename, output_frames, fps=30, loop=0)
print(f"Success! Animation saved as {output_filename}")

Loaded 5 images.
Forcing all images to master resolution: 959x782...
Rendered frame 0/90 (Pupil shifted by 0 pixels)
Rendered frame 10/90 (Pupil shifted by 17 pixels)
Rendered frame 20/90 (Pupil shifted by 32 pixels)
Rendered frame 30/90 (Pupil shifted by 43 pixels)
Rendered frame 40/90 (Pupil shifted by 49 pixels)
Rendered frame 50/90 (Pupil shifted by 49 pixels)
Rendered frame 60/90 (Pupil shifted by 43 pixels)
Rendered frame 70/90 (Pupil shifted by 32 pixels)
Rendered frame 80/90 (Pupil shifted by 17 pixels)
Saving GIF... this may take a moment.
Success! Animation saved as animated_eye_seamless.gif


### 3. Precise Iris Mapping & Eyelid Gating
- Required now to upload **target_eye_binary_mask_overlay.png** (original target eye binary mask)
- Uses a Hough transform to automatically detect and map the exact iris coordinates from the reference image.
- Applies an elliptical mask to contain the moving iris strictly within the natural eye socket.   
- Saves updated animation as **animated_iris_looking.gif**


In [5]:
from PIL import Image

# --- 1. CONFIGURATION ---
SCALE = 0.5
MAX_SHIFT_RATIO = 0.55  # Controls how far the iris slides inside the eye socket
FRAMES_PER_TRANSITION = 15

# Define the precise reference image for iris mapping
REFERENCE_IMAGE = "da9aaf73-0e8a-4007-a94a-d0e7d1499b88.png"  # or .jpg

# --- DEFINE EXACT TRANSITION IMAGES ---
required_images = [
    'qrc_eye_creator_overlay.png',
    'qrc_eye_geography_overlay.png',
    'qrc_eye_semantic_skintone.png',
    'qrc_eye_year_overlay.png',
    'qrc_eye_mosaic_soft.png'
]

image_paths = []
for filename in required_images:
    if os.path.exists(filename):
        image_paths.append(filename)
    else:
        print(f"Warning: '{filename}' not found in current directory.")

if len(image_paths) == 0:
    raise FileNotFoundError("No required overlay images found in Colab directory.")

print(f"Loaded {len(image_paths)} transition images successfully:")
for path in image_paths:
    print(f" - {path}")

# --- 2. PRECISE IRIS MAPPING FROM REFERENCE IMAGE ---
ref_img = cv2.imread(REFERENCE_IMAGE)
if ref_img is None:
    # Try JPG extension if PNG wasn't found
    ref_img = cv2.imread(REFERENCE_IMAGE.replace('.png', '.jpg'))

if ref_img is None:
    # Fallback to first image in path
    ref_img = cv2.imread(image_paths[0])

h_orig, w_orig = ref_img.shape[:2]
target_w = int(w_orig * SCALE)
target_h = int(h_orig * SCALE)

ref_scaled = cv2.resize(ref_img, (target_w, target_h))
hsv = cv2.cvtColor(ref_scaled, cv2.COLOR_BGR2HSV)

# Convert to grayscale & isolate blue/dark iris region
gray = cv2.cvtColor(ref_scaled, cv2.COLOR_BGR2GRAY)
blurred_gray = cv2.medianBlur(gray, 7)

# Detect circular iris specifically (restrict radius between 40 and 80 pixels at 0.5 scale)
circles = cv2.HoughCircles(
    blurred_gray,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=100,
    param1=50,
    param2=30,
    minRadius=int(40 * SCALE),
    maxRadius=int(120 * SCALE)
)

if circles is not None:
    circles = np.uint16(np.around(circles))
    PUPIL_X = int(circles[0][0][0])
    PUPIL_Y = int(circles[0][0][1])
    RADIUS = int(circles[0][0][2])
    print(f"✅ Precise Iris Mapped: Center=({PUPIL_X}, {PUPIL_Y}), Iris Radius={RADIUS}px")
else:
    # Hardcoded exact center of the blue iris from the reference image at 0.5 scale
    PUPIL_X = int(480 * SCALE)  # ~240px
    PUPIL_Y = int(490 * SCALE)  # ~245px
    RADIUS = int(120 * SCALE)   # ~60px
    print(f"Automatic circle fit used precise fallback Iris coordinates: Center=({PUPIL_X}, {PUPIL_Y}), Radius={RADIUS}px")

MAX_SHIFT = int(RADIUS * MAX_SHIFT_RATIO)

# --- 3. PREPARE MASKS & EYELID GATING ---
# Iris mask (Soft circular mask isolating ONLY the blue iris)
iris_mask_hard = np.zeros((target_h, target_w), dtype=np.uint8)
cv2.circle(iris_mask_hard, (PUPIL_X, PUPIL_Y), RADIUS, 255, -1)

# Inpaint mask for background sclera (White of the eye)
inpaint_mask = cv2.dilate(iris_mask_hard, np.ones((11, 11), np.uint8), iterations=2)

blurred_iris_mask = cv2.GaussianBlur(iris_mask_hard, (21, 21), 0)
mask_float = np.expand_dims(blurred_iris_mask.astype(float) / 255.0, axis=-1)

# Eyelid boundary mask (Ensures iris does not bleed over upper/lower eyelids when sliding)
sclera_mask = np.zeros((target_h, target_w), dtype=np.uint8)
# Fits an ellipse matching the opening of human eyelids
cv2.ellipse(sclera_mask, (PUPIL_X, PUPIL_Y), (int(RADIUS * 2.2), int(RADIUS * 1.2)), 0, 0, 360, 255, -1)
sclera_mask_blurred = cv2.GaussianBlur(sclera_mask, (15, 15), 0)
sclera_mask_float = np.expand_dims(sclera_mask_blurred.astype(float) / 255.0, axis=-1)

images = []
inpainted_backgrounds = []

# --- 4. PRE-INPAINT BACKGROUNDS ---
for p in image_paths:
    img = cv2.imread(p)
    if img is None: continue

    img_bgr = cv2.resize(img, (target_w, target_h))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    images.append(img_rgb)

    # Fill in the white sclera behind the iris
    bg_inpainted_bgr = cv2.inpaint(img_bgr, inpaint_mask, 20, cv2.INPAINT_TELEA)
    inpainted_backgrounds.append(cv2.cvtColor(bg_inpainted_bgr, cv2.COLOR_BGR2RGB))

num_images = len(images)
FRAMES_TOTAL = FRAMES_PER_TRANSITION * num_images
output_frames = []

# --- 5. RENDER LOOP ---
print(f"Rendering {FRAMES_TOTAL} frames...")
for i in range(FRAMES_TOTAL):
    img_idx = (i // FRAMES_PER_TRANSITION) % num_images
    next_idx = (img_idx + 1) % num_images
    local_frame = i % FRAMES_PER_TRANSITION
    alpha = local_frame / float(FRAMES_PER_TRANSITION)

    base_frame = cv2.addWeighted(images[img_idx], 1 - alpha, images[next_idx], alpha, 0)
    bg_inpainted = cv2.addWeighted(inpainted_backgrounds[img_idx], 1 - alpha, inpainted_backgrounds[next_idx], alpha, 0)

    # Half-sine wave: moves iris smoothly to the side and back center
    progress = i / float(FRAMES_TOTAL)
    shift_x = int(MAX_SHIFT * math.sin(progress * math.pi))

    M = np.float32([[1, 0, shift_x], [0, 1, 0]])

    # Shift ONLY the iris circle
    shifted_iris = cv2.warpAffine(base_frame, M, (target_w, target_h), borderMode=cv2.BORDER_REPLICATE)
    shifted_mask = cv2.warpAffine(mask_float, M, (target_w, target_h))

    if len(shifted_mask.shape) == 2:
        shifted_mask = np.expand_dims(shifted_mask, axis=-1)

    # Combine shifted iris onto the stationary white background
    eye_composite = (bg_inpainted * (1 - shifted_mask)) + (shifted_iris * shifted_mask)

    # Apply Eyelid Gate: forces everything outside the eyelids to stay stationary
    final_frame_float = (eye_composite * sclera_mask_float) + (base_frame * (1 - sclera_mask_float))
    final_frame = np.clip(final_frame_float, 0, 255).astype(np.uint8)

    output_frames.append(Image.fromarray(final_frame))

# --- 6. EXPORT ---
output_filename = 'animated_iris_looking.gif'
output_frames[0].save(
    output_filename,
    save_all=True,
    append_images=output_frames[1:],
    optimize=False,
    duration=33,
    loop=0
)
print(f"Success! Animation saved as {output_filename}")

Loaded 5 transition images successfully:
 - qrc_eye_creator_overlay.png
 - qrc_eye_geography_overlay.png
 - qrc_eye_semantic_skintone.png
 - qrc_eye_year_overlay.png
 - qrc_eye_mosaic_soft.png
✅ Precise Iris Mapped: Center=(330, 550), Iris Radius=31px
Rendering 75 frames...
Success! Animation saved as animated_iris_looking.gif


###4. Humanlike Gaze & 3D Compression
- Simulates saccadic eye movements with rapid shifts and natural fixation pauses.
- Horizontally compresses the iris as it moves to mimic the 3D perspective of a rotating eye.   
- Saves updated animation as **animated_iris_humanlike.gif**

In [6]:
import cv2
import numpy as np
import math
import glob
import os
from PIL import Image

# --- 1. CONFIGURATION ---
SCALE = 0.5
MAX_SHIFT_RATIO = 0.28  # Reduced shift distance for a subtle, natural glance
FRAMES_PER_TRANSITION = 15

REFERENCE_IMAGE = "target_eye_binary_mask_overlay.png"

# --- DEFINE EXACT TRANSITION IMAGES ---
required_images = [
    'qrc_eye_creator_overlay.png',
    'qrc_eye_geography_overlay.png',
    'qrc_eye_semantic_skintone.png',
    'qrc_eye_year_overlay.png',
    'qrc_eye_mosaic_soft.png'
]

image_paths = []
for filename in required_images:
    if os.path.exists(filename):
        image_paths.append(filename)
    else:
        print(f"Warning: '{filename}' not found in current directory.")

if len(image_paths) == 0:
    raise FileNotFoundError("No required overlay images found in Colab directory.")

print(f"Loaded {len(image_paths)} transition images successfully:")
for path in image_paths:
    print(f" - {path}")

# --- 2. PRECISE IRIS MAPPING FROM REFERENCE ---
ref_img = cv2.imread(REFERENCE_IMAGE)
if ref_img is None:
    ref_img = cv2.imread(REFERENCE_IMAGE.replace('.png', '.jpg'))
if ref_img is None:
    ref_img = cv2.imread(image_paths[0])

h_orig, w_orig = ref_img.shape[:2]
target_w = int(w_orig * SCALE)
target_h = int(h_orig * SCALE)

ref_scaled = cv2.resize(ref_img, (target_w, target_h))
gray = cv2.cvtColor(ref_scaled, cv2.COLOR_BGR2GRAY)
blurred_gray = cv2.medianBlur(gray, 7)

circles = cv2.HoughCircles(
    blurred_gray,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=100,
    param1=50,
    param2=30,
    minRadius=int(40 * SCALE),
    maxRadius=int(120 * SCALE)
)

if circles is not None:
    circles = np.uint16(np.around(circles))
    PUPIL_X = int(circles[0][0][0])
    PUPIL_Y = int(circles[0][0][1])
    RADIUS = int(circles[0][0][2])
else:
    PUPIL_X = int(480 * SCALE)
    PUPIL_Y = int(490 * SCALE)
    RADIUS = int(120 * SCALE)

MAX_SHIFT = int(RADIUS * MAX_SHIFT_RATIO)

# --- 3. PREPARE MASKS & EYELID GATING ---
iris_mask_hard = np.zeros((target_h, target_w), dtype=np.uint8)
cv2.circle(iris_mask_hard, (PUPIL_X, PUPIL_Y), RADIUS, 255, -1)

inpaint_mask = cv2.dilate(iris_mask_hard, np.ones((11, 11), np.uint8), iterations=2)

blurred_iris_mask = cv2.GaussianBlur(iris_mask_hard, (21, 21), 0)
mask_float = np.expand_dims(blurred_iris_mask.astype(float) / 255.0, axis=-1)

# Fit eyelid boundary gate
sclera_mask = np.zeros((target_h, target_w), dtype=np.uint8)
cv2.ellipse(sclera_mask, (PUPIL_X, PUPIL_Y), (int(RADIUS * 2.2), int(RADIUS * 1.2)), 0, 0, 360, 255, -1)
sclera_mask_blurred = cv2.GaussianBlur(sclera_mask, (15, 15), 0)
sclera_mask_float = np.expand_dims(sclera_mask_blurred.astype(float) / 255.0, axis=-1)

images = []
inpainted_backgrounds = []

# --- 4. PRE-INPAINT BACKGROUNDS ---
for p in image_paths:
    img = cv2.imread(p)
    if img is None: continue

    img_bgr = cv2.resize(img, (target_w, target_h))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    images.append(img_rgb)

    bg_inpainted_bgr = cv2.inpaint(img_bgr, inpaint_mask, 20, cv2.INPAINT_TELEA)
    inpainted_backgrounds.append(cv2.cvtColor(bg_inpainted_bgr, cv2.COLOR_BGR2RGB))

num_images = len(images)
FRAMES_TOTAL = FRAMES_PER_TRANSITION * num_images
output_frames = []

# --- 5. RENDER LOOP WITH SACCADIC & 3D ROTATION ---
print(f"Rendering {FRAMES_TOTAL} frames with realistic human gaze timing...")

for i in range(FRAMES_TOTAL):
    img_idx = (i // FRAMES_PER_TRANSITION) % num_images
    next_idx = (img_idx + 1) % num_images
    local_frame = i % FRAMES_PER_TRANSITION
    alpha = local_frame / float(FRAMES_PER_TRANSITION)

    base_frame = cv2.addWeighted(images[img_idx], 1 - alpha, images[next_idx], alpha, 0)
    bg_inpainted = cv2.addWeighted(inpainted_backgrounds[img_idx], 1 - alpha, inpainted_backgrounds[next_idx], alpha, 0)

    # --- SACCADIC GAZE TIMING (Fast move -> Hold -> Return) ---
    progress = i / float(FRAMES_TOTAL)

    # Smooth step / Sigmoid-style curve to simulate rapid eye movement with a fixation pause
    raw_wave = math.sin(progress * math.pi)
    # Raising to 0.6 steepens the movement curve and lengthens the hold at peak glance
    saccade_factor = math.pow(raw_wave, 0.6) if raw_wave > 0 else 0

    shift_x = int(MAX_SHIFT * saccade_factor)

    # --- 3D PERSPECTIVE COMPRESSION ---
    # As the eye rotates away from center, the iris visually narrows horizontally (foreshortening)
    compression_factor = 1.0 - (0.15 * (shift_x / float(MAX_SHIFT if MAX_SHIFT != 0 else 1)))

    # Construct Affine Matrix with horizontal scale shift anchored at pupil center
    M = np.float32([
        [compression_factor, 0, PUPIL_X * (1 - compression_factor) + shift_x],
        [0, 1, 0]
    ])

    shifted_iris = cv2.warpAffine(base_frame, M, (target_w, target_h), borderMode=cv2.BORDER_REPLICATE)
    shifted_mask = cv2.warpAffine(mask_float, M, (target_w, target_h))

    if len(shifted_mask.shape) == 2:
        shifted_mask = np.expand_dims(shifted_mask, axis=-1)

    eye_composite = (bg_inpainted * (1 - shifted_mask)) + (shifted_iris * shifted_mask)

    # Apply eyelid boundary clip
    final_frame_float = (eye_composite * sclera_mask_float) + (base_frame * (1 - sclera_mask_float))
    final_frame = np.clip(final_frame_float, 0, 255).astype(np.uint8)

    output_frames.append(Image.fromarray(final_frame))

# --- 6. EXPORT ---
output_filename = 'animated_iris_humanlike.gif'
output_frames[0].save(
    output_filename,
    save_all=True,
    append_images=output_frames[1:],
    optimize=False,
    duration=33,
    loop=0
)
print(f"Success! Realistic animation saved as {output_filename}")

Loaded 5 transition images successfully:
 - qrc_eye_creator_overlay.png
 - qrc_eye_geography_overlay.png
 - qrc_eye_semantic_skintone.png
 - qrc_eye_year_overlay.png
 - qrc_eye_mosaic_soft.png
Rendering 75 frames with realistic human gaze timing...
Success! Realistic animation saved as animated_iris_humanlike.gif


### 5. Upper Eyelid Clipping
- Refines the elliptical mask to prevent the iris from bleeding over the upper socket boundary.
- Forces the top edge of the iris to remain tucked beneath the upper eyelid.
- Saves updated animation as **animated_iris_looking_fixed.gif**  

In [7]:
import cv2
import numpy as np
import math
import glob
import os
from PIL import Image

# --- 1. CONFIGURATION ---
SCALE = 0.5
MAX_SHIFT_RATIO = 0.35  # Subtle, natural iris nudge
FRAMES_PER_TRANSITION = 15

# Define the precise reference image for iris mapping
REFERENCE_IMAGE = "target_eye_binary_mask_overlay.png"

# --- DEFINE EXACT TRANSITION IMAGES ---
required_images = [
    'qrc_eye_creator_overlay.png',
    'qrc_eye_geography_overlay.png',
    'qrc_eye_semantic_skintone.png',
    'qrc_eye_year_overlay.png',
    'qrc_eye_mosaic_soft.png'
]

image_paths = []
for filename in required_images:
    if os.path.exists(filename):
        image_paths.append(filename)
    else:
        print(f"Warning: '{filename}' not found in current directory.")

if len(image_paths) == 0:
    raise FileNotFoundError("No required overlay images found in Colab directory.")

print(f"Loaded {len(image_paths)} transition images successfully:")
for path in image_paths:
    print(f" - {path}")

# --- 2. PRECISE IRIS MAPPING FROM REFERENCE IMAGE ---
ref_img = cv2.imread(REFERENCE_IMAGE)
if ref_img is None:
    ref_img = cv2.imread(REFERENCE_IMAGE.replace('.png', '.jpg'))

if ref_img is None:
    ref_img = cv2.imread(image_paths[0])

h_orig, w_orig = ref_img.shape[:2]
target_w = int(w_orig * SCALE)
target_h = int(h_orig * SCALE)

ref_scaled = cv2.resize(ref_img, (target_w, target_h))
gray = cv2.cvtColor(ref_scaled, cv2.COLOR_BGR2GRAY)
blurred_gray = cv2.medianBlur(gray, 7)

circles = cv2.HoughCircles(
    blurred_gray,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=100,
    param1=50,
    param2=30,
    minRadius=int(40 * SCALE),
    maxRadius=int(120 * SCALE)
)

if circles is not None:
    circles = np.uint16(np.around(circles))
    PUPIL_X = int(circles[0][0][0])
    PUPIL_Y = int(circles[0][0][1])
    RADIUS = int(circles[0][0][2])
    print(f"Precise Iris Mapped: Center=({PUPIL_X}, {PUPIL_Y}), Iris Radius={RADIUS}px")
else:
    PUPIL_X = int(480 * SCALE)  # ~240px
    PUPIL_Y = int(490 * SCALE)  # ~245px
    RADIUS = int(120 * SCALE)   # ~60px
    print(f"Using fallback Iris coordinates: Center=({PUPIL_X}, {PUPIL_Y}), Radius={RADIUS}px")

MAX_SHIFT = int(RADIUS * MAX_SHIFT_RATIO)

# --- 3. PREPARE MASKS & TIGHT EYELID GATING ---
iris_mask_hard = np.zeros((target_h, target_w), dtype=np.uint8)
cv2.circle(iris_mask_hard, (PUPIL_X, PUPIL_Y), RADIUS, 255, -1)

# Inpaint mask for background sclera
inpaint_mask = cv2.dilate(iris_mask_hard, np.ones((11, 11), np.uint8), iterations=2)

blurred_iris_mask = cv2.GaussianBlur(iris_mask_hard, (15, 15), 0)
mask_float = np.expand_dims(blurred_iris_mask.astype(float) / 255.0, axis=-1)

# EYELID GATING FIX: Vertical height reduced to 0.85 * RADIUS to stop upper eyelid overlap
sclera_mask = np.zeros((target_h, target_w), dtype=np.uint8)
eyelid_center_y = PUPIL_Y + int(5 * SCALE)  # Slight downward offset to clear upper lashes
cv2.ellipse(sclera_mask, (PUPIL_X, eyelid_center_y), (int(RADIUS * 2.0), int(RADIUS * 0.85)), 0, 0, 360, 255, -1)

sclera_mask_blurred = cv2.GaussianBlur(sclera_mask, (11, 11), 0)
sclera_mask_float = np.expand_dims(sclera_mask_blurred.astype(float) / 255.0, axis=-1)

images = []
inpainted_backgrounds = []

# --- 4. PRE-INPAINT BACKGROUNDS ---
for p in image_paths:
    img = cv2.imread(p)
    if img is None: continue

    img_bgr = cv2.resize(img, (target_w, target_h))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    images.append(img_rgb)

    bg_inpainted_bgr = cv2.inpaint(img_bgr, inpaint_mask, 20, cv2.INPAINT_TELEA)
    inpainted_backgrounds.append(cv2.cvtColor(bg_inpainted_bgr, cv2.COLOR_BGR2RGB))

num_images = len(images)
FRAMES_TOTAL = FRAMES_PER_TRANSITION * num_images
output_frames = []

# --- 5. RENDER LOOP ---
print(f"Rendering {FRAMES_TOTAL} frames with tight upper eyelid clipping...")
for i in range(FRAMES_TOTAL):
    img_idx = (i // FRAMES_PER_TRANSITION) % num_images
    next_idx = (img_idx + 1) % num_images
    local_frame = i % FRAMES_PER_TRANSITION
    alpha = local_frame / float(FRAMES_PER_TRANSITION)

    base_frame = cv2.addWeighted(images[img_idx], 1 - alpha, images[next_idx], alpha, 0)
    bg_inpainted = cv2.addWeighted(inpainted_backgrounds[img_idx], 1 - alpha, inpainted_backgrounds[next_idx], alpha, 0)

    progress = i / float(FRAMES_TOTAL)
    shift_x = int(MAX_SHIFT * math.sin(progress * math.pi))

    M = np.float32([[1, 0, shift_x], [0, 1, 0]])

    shifted_iris = cv2.warpAffine(base_frame, M, (target_w, target_h), borderMode=cv2.BORDER_REPLICATE)
    shifted_mask = cv2.warpAffine(mask_float, M, (target_w, target_h))

    if len(shifted_mask.shape) == 2:
        shifted_mask = np.expand_dims(shifted_mask, axis=-1)

    eye_composite = (bg_inpainted * (1 - shifted_mask)) + (shifted_iris * shifted_mask)

    # Eyelid clipping: forces the top edge of the iris to stay tucked under the upper lid
    final_frame_float = (eye_composite * sclera_mask_float) + (base_frame * (1 - sclera_mask_float))
    final_frame = np.clip(final_frame_float, 0, 255).astype(np.uint8)

    output_frames.append(Image.fromarray(final_frame))

# --- 6. EXPORT ---
output_filename = 'animated_iris_looking_fixed.gif'
output_frames[0].save(
    output_filename,
    save_all=True,
    append_images=output_frames[1:],
    optimize=False,
    duration=33,
    loop=0
)
print(f"Success! Animation saved as {output_filename}")

Loaded 5 transition images successfully:
 - qrc_eye_creator_overlay.png
 - qrc_eye_geography_overlay.png
 - qrc_eye_semantic_skintone.png
 - qrc_eye_year_overlay.png
 - qrc_eye_mosaic_soft.png
Precise Iris Mapped: Center=(184, 150), Iris Radius=44px
Rendering 75 frames with tight upper eyelid clipping...
Success! Animation saved as animated_iris_looking_fixed.gif


###6. Ghost-Free Sclera Fill
- Aggressively dilates the erasure mask to remove static iris shadows or residue.
- Fills the background void with sampled clean sclera colour to prevent trailing smudges during movement.   
- Saves updated animation as **animated_iris_ghost_free.gif**

In [8]:
import cv2
import numpy as np
import math
import glob
from PIL import Image

# --- 1. CONFIGURATION ---
SCALE = 0.5
MAX_SHIFT_RATIO = 0.35
FRAMES_PER_TRANSITION = 15

REFERENCE_IMAGE = "target_eye_binary_mask_overlay.png"

# --- DEFINE EXACT TRANSITION IMAGES ---
required_images = [
    'qrc_eye_creator_overlay.png',
    'qrc_eye_geography_overlay.png',
    'qrc_eye_semantic_skintone.png',
    'qrc_eye_year_overlay.png',
    'qrc_eye_mosaic_soft.png'
]

image_paths = []
for filename in required_images:
    if os.path.exists(filename):
        image_paths.append(filename)
    else:
        print(f"Warning: '{filename}' not found in current directory.")

if len(image_paths) == 0:
    raise FileNotFoundError("No required overlay images found in Colab directory.")

print(f"Loaded {len(image_paths)} transition images successfully:")
for path in image_paths:
    print(f" - {path}")

# --- 2. PRECISE IRIS MAPPING FROM REFERENCE ---
ref_img = cv2.imread(REFERENCE_IMAGE)
if ref_img is None:
    ref_img = cv2.imread(REFERENCE_IMAGE.replace('.png', '.jpg'))
if ref_img is None:
    ref_img = cv2.imread(image_paths[0])

h_orig, w_orig = ref_img.shape[:2]
target_w = int(w_orig * SCALE)
target_h = int(h_orig * SCALE)

ref_scaled = cv2.resize(ref_img, (target_w, target_h))
gray = cv2.cvtColor(ref_scaled, cv2.COLOR_BGR2GRAY)
blurred_gray = cv2.medianBlur(gray, 7)

circles = cv2.HoughCircles(
    blurred_gray,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=100,
    param1=50,
    param2=30,
    minRadius=int(40 * SCALE),
    maxRadius=int(120 * SCALE)
)

if circles is not None:
    circles = np.uint16(np.around(circles))
    PUPIL_X = int(circles[0][0][0])
    PUPIL_Y = int(circles[0][0][1])
    RADIUS = int(circles[0][0][2])
else:
    PUPIL_X = int(480 * SCALE)
    PUPIL_Y = int(490 * SCALE)
    RADIUS = int(120 * SCALE)

MAX_SHIFT = int(RADIUS * MAX_SHIFT_RATIO)

# --- 3. MASKS & CLEAN SCLERA FILL ---
iris_mask_hard = np.zeros((target_h, target_w), dtype=np.uint8)
cv2.circle(iris_mask_hard, (PUPIL_X, PUPIL_Y), RADIUS, 255, -1)

# Crisp iris mask to avoid trailing blur
blurred_iris_mask = cv2.GaussianBlur(iris_mask_hard, (3, 3), 0)
mask_float = np.expand_dims(blurred_iris_mask.astype(float) / 255.0, axis=-1)

# Eyelid gate
sclera_mask = np.zeros((target_h, target_w), dtype=np.uint8)
eyelid_center_y = PUPIL_Y + int(5 * SCALE)
cv2.ellipse(sclera_mask, (PUPIL_X, eyelid_center_y), (int(RADIUS * 2.0), int(RADIUS * 0.82)), 0, 0, 360, 255, -1)
sclera_mask_blurred = cv2.GaussianBlur(sclera_mask, (5, 5), 0)
sclera_mask_float = np.expand_dims(sclera_mask_blurred.astype(float) / 255.0, axis=-1)

images = []
clean_backgrounds = []

# Dilated mask used for cleaning out dark iris pixels entirely
clean_erase_mask = cv2.dilate(iris_mask_hard, np.ones((15, 15), np.uint8), iterations=2)

# --- 4. PRE-CLEAN BACKGROUNDS (NO SMUDGING) ---
for p in image_paths:
    img = cv2.imread(p)
    if img is None: continue

    img_bgr = cv2.resize(img, (target_w, target_h))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    images.append(img_rgb)

    # First pass inpaint
    bg_inpainted = cv2.inpaint(img_bgr, clean_erase_mask, 25, cv2.INPAINT_TELEA)

    # Sample actual white sclera color adjacent to iris (to the left)
    sclera_sample_x = max(0, PUPIL_X - RADIUS - 15)
    sclera_color = bg_inpainted[PUPIL_Y, sclera_sample_x].astype(float)

    # Fill the erased iris area with the clean sclera color to guarantee NO dark ghosting
    bg_clean = bg_inpainted.copy().astype(float)
    erase_float = np.expand_dims(cv2.GaussianBlur(clean_erase_mask, (11, 11), 0).astype(float) / 255.0, axis=-1)

    bg_clean = (bg_clean * (1 - erase_float)) + (sclera_color * erase_float)
    clean_backgrounds.append(cv2.cvtColor(np.clip(bg_clean, 0, 255).astype(np.uint8), cv2.COLOR_BGR2RGB))

num_images = len(images)
FRAMES_TOTAL = FRAMES_PER_TRANSITION * num_images
output_frames = []

# --- 5. RENDER LOOP ---
print(f"Rendering {FRAMES_TOTAL} frames with clean background fill...")
for i in range(FRAMES_TOTAL):
    img_idx = (i // FRAMES_PER_TRANSITION) % num_images
    next_idx = (img_idx + 1) % num_images
    local_frame = i % FRAMES_PER_TRANSITION
    alpha = local_frame / float(FRAMES_PER_TRANSITION)

    base_frame = cv2.addWeighted(images[img_idx], 1 - alpha, images[next_idx], alpha, 0)
    bg_clean = cv2.addWeighted(clean_backgrounds[img_idx], 1 - alpha, clean_backgrounds[next_idx], alpha, 0)

    progress = i / float(FRAMES_TOTAL)
    shift_x = int(MAX_SHIFT * math.sin(progress * math.pi))

    M = np.float32([[1, 0, shift_x], [0, 1, 0]])

    shifted_iris = cv2.warpAffine(base_frame, M, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    shifted_mask = cv2.warpAffine(mask_float, M, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    if len(shifted_mask.shape) == 2:
        shifted_mask = np.expand_dims(shifted_mask, axis=-1)

    # Overlay crisp shifted iris onto pure clean sclera
    eye_composite = (bg_clean * (1 - shifted_mask)) + (shifted_iris * shifted_mask)

    # Clip cleanly behind eyelids
    final_frame_float = (eye_composite * sclera_mask_float) + (base_frame * (1 - sclera_mask_float))
    final_frame = np.clip(final_frame_float, 0, 255).astype(np.uint8)

    output_frames.append(Image.fromarray(final_frame))

# --- 6. EXPORT ---
output_filename = 'animated_iris_ghost_free.gif'
output_frames[0].save(
    output_filename,
    save_all=True,
    append_images=output_frames[1:],
    optimize=False,
    duration=33,
    loop=0
)
print(f"Success! Ghost-free animation saved as {output_filename}")

Loaded 5 transition images successfully:
 - qrc_eye_creator_overlay.png
 - qrc_eye_geography_overlay.png
 - qrc_eye_semantic_skintone.png
 - qrc_eye_year_overlay.png
 - qrc_eye_mosaic_soft.png
Rendering 75 frames with clean background fill...
Success! Ghost-free animation saved as animated_iris_ghost_free.gif


###7. Uniform Sclera Standardisation
- Replaces the underlying eye socket with a synthetic, pure white background disc.
- Removes any vignetting, shadow variations, or uneven lighting artefacts present in the generated source images.   
- Saves updated image as **animated_iris_clean_white_disc.gif**

In [9]:
import cv2
import numpy as np
import math
import glob
from PIL import Image

# --- 1. CONFIGURATION ---
SCALE = 0.5
MAX_SHIFT_RATIO = 0.35  # Subtle, natural nudge
FRAMES_PER_TRANSITION = 15

REFERENCE_IMAGE = "target_eye_binary_mask_overlay.png"

# --- DEFINE EXACT TRANSITION IMAGES ---
required_images = [
    'qrc_eye_creator_overlay.png',
    'qrc_eye_geography_overlay.png',
    'qrc_eye_semantic_skintone.png',
    'qrc_eye_year_overlay.png',
    'qrc_eye_mosaic_soft.png'
]

image_paths = []
for filename in required_images:
    if os.path.exists(filename):
        image_paths.append(filename)
    else:
        print(f"Warning: '{filename}' not found in current directory.")

if len(image_paths) == 0:
    raise FileNotFoundError("No required overlay images found in Colab directory.")

print(f"Loaded {len(image_paths)} transition images successfully:")
for path in image_paths:
    print(f" - {path}")

# --- 2. PRECISE IRIS MAPPING FROM REFERENCE ---
ref_img = cv2.imread(REFERENCE_IMAGE)
if ref_img is None:
    ref_img = cv2.imread(REFERENCE_IMAGE.replace('.png', '.jpg'))
if ref_img is None:
    ref_img = cv2.imread(image_paths[0])

h_orig, w_orig = ref_img.shape[:2]
target_w = int(w_orig * SCALE)
target_h = int(h_orig * SCALE)

ref_scaled = cv2.resize(ref_img, (target_w, target_h))
gray = cv2.cvtColor(ref_scaled, cv2.COLOR_BGR2GRAY)
blurred_gray = cv2.medianBlur(gray, 7)

circles = cv2.HoughCircles(
    blurred_gray,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=100,
    param1=50,
    param2=30,
    minRadius=int(40 * SCALE),
    maxRadius=int(120 * SCALE)
)

if circles is not None:
    circles = np.uint16(np.around(circles))
    PUPIL_X = int(circles[0][0][0])
    PUPIL_Y = int(circles[0][0][1])
    RADIUS = int(circles[0][0][2])
else:
    PUPIL_X = int(480 * SCALE)
    PUPIL_Y = int(490 * SCALE)
    RADIUS = int(120 * SCALE)

MAX_SHIFT = int(RADIUS * MAX_SHIFT_RATIO)

# --- 3. MASKS & PURE WHITE SCLERA DISC ---
iris_mask_hard = np.zeros((target_h, target_w), dtype=np.uint8)
cv2.circle(iris_mask_hard, (PUPIL_X, PUPIL_Y), RADIUS, 255, -1)

# Crisp iris mask (3x3 blur) stops trailing smears
blurred_iris_mask = cv2.GaussianBlur(iris_mask_hard, (3, 3), 0)
mask_float = np.expand_dims(blurred_iris_mask.astype(float) / 255.0, axis=-1)

# Full Eyelid / Sclera Gate
sclera_mask = np.zeros((target_h, target_w), dtype=np.uint8)
eyelid_center_y = PUPIL_Y + int(5 * SCALE)
cv2.ellipse(sclera_mask, (PUPIL_X, eyelid_center_y), (int(RADIUS * 2.1), int(RADIUS * 0.85)), 0, 0, 360, 255, -1)
sclera_mask_blurred = cv2.GaussianBlur(sclera_mask, (7, 7), 0)
sclera_mask_float = np.expand_dims(sclera_mask_blurred.astype(float) / 255.0, axis=-1)

# PURE WHITE DISC FIX: Generates a clean, uniform white canvas (250, 250, 250) for the eye socket
white_disc_canvas = np.full((target_h, target_w, 3), 250, dtype=np.uint8)

# Expanded inpaint mask to 100% erase the static iris underneath
inpaint_mask_large = cv2.dilate(iris_mask_hard, np.ones((25, 25), np.uint8), iterations=3)

images = []
inpainted_backgrounds = []

# --- 4. PRE-INPAINT BACKGROUNDS ---
for p in image_paths:
    img = cv2.imread(p)
    if img is None: continue

    img_bgr = cv2.resize(img, (target_w, target_h))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    images.append(img_rgb)

    # Completely erase static iris from source images
    bg_inpainted_bgr = cv2.inpaint(img_bgr, inpaint_mask_large, 30, cv2.INPAINT_TELEA)
    bg_inpainted_rgb = cv2.cvtColor(bg_inpainted_bgr, cv2.COLOR_BGR2RGB)

    # Blend pure white disc into the eye opening to remove all vignetting / patch artifacts
    bg_clean = (bg_inpainted_rgb.astype(float) * (1 - sclera_mask_float)) + (white_disc_canvas.astype(float) * sclera_mask_float)
    inpainted_backgrounds.append(np.clip(bg_clean, 0, 255).astype(np.uint8))

num_images = len(images)
FRAMES_TOTAL = FRAMES_PER_TRANSITION * num_images
output_frames = []

# --- 5. RENDER LOOP ---
print(f"Rendering {FRAMES_TOTAL} frames with white sclera disc...")
for i in range(FRAMES_TOTAL):
    img_idx = (i // FRAMES_PER_TRANSITION) % num_images
    next_idx = (img_idx + 1) % num_images
    local_frame = i % FRAMES_PER_TRANSITION
    alpha = local_frame / float(FRAMES_PER_TRANSITION)

    base_frame = cv2.addWeighted(images[img_idx], 1 - alpha, images[next_idx], alpha, 0)
    bg_clean = cv2.addWeighted(inpainted_backgrounds[img_idx], 1 - alpha, inpainted_backgrounds[next_idx], alpha, 0)

    progress = i / float(FRAMES_TOTAL)
    shift_x = int(MAX_SHIFT * math.sin(progress * math.pi))

    M = np.float32([[1, 0, shift_x], [0, 1, 0]])

    shifted_iris = cv2.warpAffine(base_frame, M, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    shifted_mask = cv2.warpAffine(mask_float, M, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    if len(shifted_mask.shape) == 2:
        shifted_mask = np.expand_dims(shifted_mask, axis=-1)

    # Place crisp shifted iris over clean white sclera background
    eye_composite = (bg_clean * (1 - shifted_mask)) + (shifted_iris * shifted_mask)

    # Clip cleanly inside eyelid boundaries
    final_frame_float = (eye_composite * sclera_mask_float) + (base_frame * (1 - sclera_mask_float))
    final_frame = np.clip(final_frame_float, 0, 255).astype(np.uint8)

    output_frames.append(Image.fromarray(final_frame))

# --- 6. EXPORT ---
output_filename = 'animated_iris_clean_white_disc.gif'
output_frames[0].save(
    output_filename,
    save_all=True,
    append_images=output_frames[1:],
    optimize=False,
    duration=33,
    loop=0
)
print(f"Success! Fixed animation saved as {output_filename}")

Loaded 5 transition images successfully:
 - qrc_eye_creator_overlay.png
 - qrc_eye_geography_overlay.png
 - qrc_eye_semantic_skintone.png
 - qrc_eye_year_overlay.png
 - qrc_eye_mosaic_soft.png
Rendering 75 frames with white sclera disc...
Success! Fixed animation saved as animated_iris_clean_white_disc.gif


###8. Canvas Colour Match & Lower Expansion
- Expands the eyelid mask downwards to cover the full lower socket and lash line.
- Samples the background canvas tone to smoothly blend the outer edges of the eye opening.   
- Updated animation saved as **animated_iris_full_coverage.gif**

In [10]:
import cv2
import numpy as np
import math
import glob
from PIL import Image

# --- 1. CONFIGURATION ---
SCALE = 0.5
MAX_SHIFT_RATIO = 0.35  # Subtle, natural nudge
FRAMES_PER_TRANSITION = 15

REFERENCE_IMAGE = "target_eye_binary_mask_overlay.png"

# --- DEFINE EXACT TRANSITION IMAGES ---
required_images = [
    'qrc_eye_creator_overlay.png',
    'qrc_eye_geography_overlay.png',
    'qrc_eye_semantic_skintone.png',
    'qrc_eye_year_overlay.png',
    'qrc_eye_mosaic_soft.png'
]

image_paths = []
for filename in required_images:
    if os.path.exists(filename):
        image_paths.append(filename)
    else:
        print(f"Warning: '{filename}' not found in current directory.")

if len(image_paths) == 0:
    raise FileNotFoundError("No required overlay images found in Colab directory.")

print(f"Loaded {len(image_paths)} transition images successfully:")
for path in image_paths:
    print(f" - {path}")

# --- 2. PRECISE IRIS MAPPING FROM REFERENCE IMAGE ---
ref_img = cv2.imread(REFERENCE_IMAGE)
if ref_img is None:
    ref_img = cv2.imread(REFERENCE_IMAGE.replace('.png', '.jpg'))
if ref_img is None:
    ref_img = cv2.imread(image_paths[0])

h_orig, w_orig = ref_img.shape[:2]
target_w = int(w_orig * SCALE)
target_h = int(h_orig * SCALE)

ref_scaled = cv2.resize(ref_img, (target_w, target_h))
gray = cv2.cvtColor(ref_scaled, cv2.COLOR_BGR2GRAY)
blurred_gray = cv2.medianBlur(gray, 7)

circles = cv2.HoughCircles(
    blurred_gray,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=100,
    param1=50,
    param2=30,
    minRadius=int(40 * SCALE),
    maxRadius=int(120 * SCALE)
)

if circles is not None:
    circles = np.uint16(np.around(circles))
    PUPIL_X = int(circles[0][0][0])
    PUPIL_Y = int(circles[0][0][1])
    RADIUS = int(circles[0][0][2])
else:
    PUPIL_X = int(480 * SCALE)
    PUPIL_Y = int(490 * SCALE)
    RADIUS = int(120 * SCALE)

MAX_SHIFT = int(RADIUS * MAX_SHIFT_RATIO)

# --- 3. MASKS & CANVAS COLOR FILL ---
# Moving Iris Mask
iris_mask_hard = np.zeros((target_h, target_w), dtype=np.uint8)
cv2.circle(iris_mask_hard, (PUPIL_X, PUPIL_Y), RADIUS, 255, -1)

# Crisp iris edge to prevent motion blur
blurred_iris_mask = cv2.GaussianBlur(iris_mask_hard, (3, 3), 0)
mask_float = np.expand_dims(blurred_iris_mask.astype(float) / 255.0, axis=-1)

# Balanced Eyelid Gate: Fits upper lid tightly and extends down to lower lash line
sclera_mask = np.zeros((target_h, target_w), dtype=np.uint8)
eyelid_center_y = PUPIL_Y + int(8 * SCALE)  # Shifted down slightly to cover lower socket
cv2.ellipse(sclera_mask, (PUPIL_X, eyelid_center_y), (int(RADIUS * 1.95), int(RADIUS * 1.05)), 0, 0, 360, 255, -1)
sclera_mask_blurred = cv2.GaussianBlur(sclera_mask, (7, 7), 0)
sclera_mask_float = np.expand_dims(sclera_mask_blurred.astype(float) / 255.0, axis=-1)

# Sample background canvas color from top corner to prevent off-white patches
sample_color_bgr = ref_scaled[10, 10]
canvas_fill_rgb = np.array([sample_color_bgr[2], sample_color_bgr[1], sample_color_bgr[0]], dtype=float)

# Expanded inpaint mask to eliminate all dark iris residue underneath
inpaint_mask_large = cv2.dilate(iris_mask_hard, np.ones((21, 21), np.uint8), iterations=2)

images = []
inpainted_backgrounds = []

# --- 4. PRE-INPAINT BACKGROUNDS ---
for p in image_paths:
    img = cv2.imread(p)
    if img is None: continue

    img_bgr = cv2.resize(img, (target_w, target_h))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    images.append(img_rgb)

    # Inpaint static iris area
    bg_inpainted_bgr = cv2.inpaint(img_bgr, inpaint_mask_large, 25, cv2.INPAINT_TELEA)
    bg_inpainted_rgb = cv2.cvtColor(bg_inpainted_bgr, cv2.COLOR_BGR2RGB).astype(float)

    # Blend canvas-matched color into inpaint zone to prevent patchiness
    inpaint_float = np.expand_dims(cv2.GaussianBlur(inpaint_mask_large, (11, 11), 0).astype(float) / 255.0, axis=-1)
    bg_clean = (bg_inpainted_rgb * (1 - inpaint_float)) + (canvas_fill_rgb * inpaint_float)

    inpainted_backgrounds.append(np.clip(bg_clean, 0, 255).astype(np.uint8))

num_images = len(images)
FRAMES_TOTAL = FRAMES_PER_TRANSITION * num_images
output_frames = []

# --- 5. RENDER LOOP (CORRECT IRIS MOTION) ---
print(f"Rendering {FRAMES_TOTAL} frames with clean iris movement...")
for i in range(FRAMES_TOTAL):
    img_idx = (i // FRAMES_PER_TRANSITION) % num_images
    next_idx = (img_idx + 1) % num_images
    local_frame = i % FRAMES_PER_TRANSITION
    alpha = local_frame / float(FRAMES_PER_TRANSITION)

    base_frame = cv2.addWeighted(images[img_idx], 1 - alpha, images[next_idx], alpha, 0)
    bg_inpainted = cv2.addWeighted(inpainted_backgrounds[img_idx], 1 - alpha, inpainted_backgrounds[next_idx], alpha, 0)

    progress = i / float(FRAMES_TOTAL)
    shift_x = int(MAX_SHIFT * math.sin(progress * math.pi))

    M = np.float32([[1, 0, shift_x], [0, 1, 0]])

    # Shift ONLY the iris circle
    shifted_iris = cv2.warpAffine(base_frame, M, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    shifted_mask = cv2.warpAffine(mask_float, M, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    if len(shifted_mask.shape) == 2:
        shifted_mask = np.expand_dims(shifted_mask, axis=-1)

    # Place shifted iris over clean background socket
    eye_composite = (bg_inpainted * (1 - shifted_mask)) + (shifted_iris * shifted_mask)

    # Eyelid gate clipping
    final_frame_float = (eye_composite * sclera_mask_float) + (base_frame * (1 - sclera_mask_float))
    final_frame = np.clip(final_frame_float, 0, 255).astype(np.uint8)

    output_frames.append(Image.fromarray(final_frame))

# --- 6. EXPORT ---
output_filename = 'animated_iris_full_coverage.gif'
output_frames[0].save(
    output_filename,
    save_all=True,
    append_images=output_frames[1:],
    optimize=False,
    duration=33,
    loop=0
)
print(f"Success! Corrected animation saved as {output_filename}")

Loaded 5 transition images successfully:
 - qrc_eye_creator_overlay.png
 - qrc_eye_geography_overlay.png
 - qrc_eye_semantic_skintone.png
 - qrc_eye_year_overlay.png
 - qrc_eye_mosaic_soft.png
Rendering 75 frames with clean iris movement...
Success! Corrected animation saved as animated_iris_full_coverage.gif


### 9. Dual-Eye Synchronisation
- Horizontally flips the base frames and masks to construct a mirrored left eye.
- Enforces parallel tracking motion for both irises and horizontally stitches the two halves together to complete the dynamic artwork.
- Saves final GIF file artwork as **animated_pair_of_eyes_natural.gif**

In [11]:
import cv2
import numpy as np
import math
import glob
from PIL import Image

# --- 1. CONFIGURATION ---
SCALE = 0.5
MAX_SHIFT_RATIO = 0.35
FRAMES_PER_TRANSITION = 15

REFERENCE_IMAGE = "target_eye_binary_mask_overlay.png"

# --- DEFINE EXACT TRANSITION IMAGES ---
required_images = [
    'qrc_eye_creator_overlay.png',
    'qrc_eye_geography_overlay.png',
    'qrc_eye_semantic_skintone.png',
    'qrc_eye_year_overlay.png',
    'qrc_eye_mosaic_soft.png'
]

image_paths = []
for filename in required_images:
    if os.path.exists(filename):
        image_paths.append(filename)
    else:
        print(f"Warning: '{filename}' not found in current directory.")

if len(image_paths) == 0:
    raise FileNotFoundError("No required overlay images found in Colab directory.")

print(f"Loaded {len(image_paths)} transition images successfully:")
for path in image_paths:
    print(f" - {path}")
# --- 2. PRECISE IRIS MAPPING FROM REFERENCE ---
ref_img = cv2.imread(REFERENCE_IMAGE)
if ref_img is None:
    ref_img = cv2.imread(REFERENCE_IMAGE.replace('.png', '.jpg'))
if ref_img is None:
    ref_img = cv2.imread(image_paths[0])

h_orig, w_orig = ref_img.shape[:2]
target_w = int(w_orig * SCALE)
target_h = int(h_orig * SCALE)

ref_scaled = cv2.resize(ref_img, (target_w, target_h))
gray = cv2.cvtColor(ref_scaled, cv2.COLOR_BGR2GRAY)
blurred_gray = cv2.medianBlur(gray, 7)

circles = cv2.HoughCircles(
    blurred_gray, cv2.HOUGH_GRADIENT, dp=1, minDist=100,
    param1=50, param2=30, minRadius=int(40 * SCALE), maxRadius=int(120 * SCALE)
)

if circles is not None:
    circles = np.uint16(np.around(circles))
    PUPIL_X = int(circles[0][0][0])
    PUPIL_Y = int(circles[0][0][1])
    RADIUS = int(circles[0][0][2])
else:
    PUPIL_X = int(480 * SCALE)
    PUPIL_Y = int(490 * SCALE)
    RADIUS = int(120 * SCALE)

MAX_SHIFT = int(RADIUS * MAX_SHIFT_RATIO)

# --- 3. MASKS & CANVAS COLOR FILL ---
iris_mask_hard = np.zeros((target_h, target_w), dtype=np.uint8)
cv2.circle(iris_mask_hard, (PUPIL_X, PUPIL_Y), RADIUS, 255, -1)

blurred_iris_mask = cv2.GaussianBlur(iris_mask_hard, (3, 3), 0)
mask_float = np.expand_dims(blurred_iris_mask.astype(float) / 255.0, axis=-1)

sclera_mask = np.zeros((target_h, target_w), dtype=np.uint8)
eyelid_center_y = PUPIL_Y + int(8 * SCALE)
cv2.ellipse(sclera_mask, (PUPIL_X, eyelid_center_y), (int(RADIUS * 1.95), int(RADIUS * 1.05)), 0, 0, 360, 255, -1)
sclera_mask_blurred = cv2.GaussianBlur(sclera_mask, (7, 7), 0)
sclera_mask_float = np.expand_dims(sclera_mask_blurred.astype(float) / 255.0, axis=-1)

sample_color_bgr = ref_scaled[10, 10]
canvas_fill_rgb = np.array([sample_color_bgr[2], sample_color_bgr[1], sample_color_bgr[0]], dtype=float)

inpaint_mask_large = cv2.dilate(iris_mask_hard, np.ones((21, 21), np.uint8), iterations=2)

images = []
inpainted_backgrounds = []

for p in image_paths:
    img = cv2.imread(p)
    if img is None: continue

    img_bgr = cv2.resize(img, (target_w, target_h))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    images.append(img_rgb)

    bg_inpainted_bgr = cv2.inpaint(img_bgr, inpaint_mask_large, 25, cv2.INPAINT_TELEA)
    bg_inpainted_rgb = cv2.cvtColor(bg_inpainted_bgr, cv2.COLOR_BGR2RGB).astype(float)

    inpaint_float = np.expand_dims(cv2.GaussianBlur(inpaint_mask_large, (11, 11), 0).astype(float) / 255.0, axis=-1)
    bg_clean = (bg_inpainted_rgb * (1 - inpaint_float)) + (canvas_fill_rgb * inpaint_float)

    inpainted_backgrounds.append(np.clip(bg_clean, 0, 255).astype(np.uint8))

num_images = len(images)
FRAMES_TOTAL = FRAMES_PER_TRANSITION * num_images
output_frames = []

# --- 4. RENDER LOOP (NATURAL FACIAL LAYOUT & PARALLEL GAZE) ---
print(f"Rendering {FRAMES_TOTAL} dual-eye frames...")

for i in range(FRAMES_TOTAL):
    img_idx = (i // FRAMES_PER_TRANSITION) % num_images
    next_idx = (img_idx + 1) % num_images
    local_frame = i % FRAMES_PER_TRANSITION
    alpha = local_frame / float(FRAMES_PER_TRANSITION)

    base_frame = cv2.addWeighted(images[img_idx], 1 - alpha, images[next_idx], alpha, 0)
    bg_inpainted = cv2.addWeighted(inpainted_backgrounds[img_idx], 1 - alpha, inpainted_backgrounds[next_idx], alpha, 0)

    progress = i / float(FRAMES_TOTAL)
    shift_x = int(MAX_SHIFT * math.sin(progress * math.pi))

    # --- RIGHT EYE OF FACE (Original Image) ---
    M_right = np.float32([[1, 0, shift_x], [0, 1, 0]])
    iris_right = cv2.warpAffine(base_frame, M_right, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT)
    mask_right = cv2.warpAffine(mask_float, M_right, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT)
    if len(mask_right.shape) == 2:
        mask_right = np.expand_dims(mask_right, axis=-1)

    eye_right_comp = (bg_inpainted * (1 - mask_right)) + (iris_right * mask_right)
    right_eye_float = (eye_right_comp * sclera_mask_float) + (base_frame * (1 - sclera_mask_float))
    right_eye = np.clip(right_eye_float, 0, 255).astype(np.uint8)

    # --- LEFT EYE OF FACE (Flipped Image + Same Motion Direction) ---
    base_frame_left = cv2.flip(base_frame, 1)
    bg_inpainted_left = cv2.flip(bg_inpainted, 1)

    # MOTION FIX: Keep shift_x identical so both pupils track in parallel
    shift_x_left = shift_x
    M_left = np.float32([[1, 0, shift_x_left], [0, 1, 0]])

    mask_float_left = cv2.flip(mask_float, 1)
    sclera_mask_float_left = cv2.flip(sclera_mask_float, 1)

    if len(mask_float_left.shape) == 2:
        mask_float_left = np.expand_dims(mask_float_left, axis=-1)
    if len(sclera_mask_float_left.shape) == 2:
        sclera_mask_float_left = np.expand_dims(sclera_mask_float_left, axis=-1)

    iris_left = cv2.warpAffine(base_frame_left, M_left, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT)
    mask_left = cv2.warpAffine(mask_float_left, M_left, (target_w, target_h), borderMode=cv2.BORDER_CONSTANT)
    if len(mask_left.shape) == 2:
        mask_left = np.expand_dims(mask_left, axis=-1)

    eye_left_comp = (bg_inpainted_left * (1 - mask_left)) + (iris_left * mask_left)
    left_eye_float = (eye_left_comp * sclera_mask_float_left) + (base_frame_left * (1 - sclera_mask_float_left))
    left_eye = np.clip(left_eye_float, 0, 255).astype(np.uint8)

    # LAYOUT FIX: Put left eye graphic on left side of image (right_eye graphic on right side)
    dual_eyes = np.hstack((right_eye, left_eye))
    output_frames.append(Image.fromarray(dual_eyes))

# --- 5. EXPORT ---
output_filename = 'animated_pair_of_eyes_natural.gif'
output_frames[0].save(
    output_filename, save_all=True, append_images=output_frames[1:],
    optimize=False, duration=33, loop=0
)
print(f"Success! Natural pair of eyes saved as {output_filename}")

Loaded 5 transition images successfully:
 - qrc_eye_creator_overlay.png
 - qrc_eye_geography_overlay.png
 - qrc_eye_semantic_skintone.png
 - qrc_eye_year_overlay.png
 - qrc_eye_mosaic_soft.png
Rendering 75 dual-eye frames...
Success! Natural pair of eyes saved as animated_pair_of_eyes_natural.gif
